# Stage 2: Full Non-TA Feature Pool Test Runner

Run the setup cell first, then run each model cell separately.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd()
if not (project_root / "models").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

project_root

## Prepare Data

Creates or refreshes:

- `data-folds-full-non-ta` for Ridge, XGBoost, LightGBM, AutoGluon
- `data-folds-full-non-ta-nn` for LSTM with train-only MinMax scaling

In [ ]:
from models.full_non_ta_feature_pool import create_full_non_ta_folds, create_scaled_full_non_ta_nn_folds

tabular_data_dir = create_full_non_ta_folds()
lstm_data_dir = create_scaled_full_non_ta_nn_folds()

{
    "tabular_data": tabular_data_dir,
    "lstm_scaled_data": lstm_data_dir,
}

## Ridge Regression

Data: `data-folds-full-non-ta`

Scaler: `StandardScaler` fitted on each train fold only

In [ ]:
from models.full_non_ta_experiments import run_ridge_full_non_ta

ridge_metrics = run_ridge_full_non_ta()
ridge_metrics

## XGBoost

Data: `data-folds-full-non-ta`

Scaler: none

In [ ]:
from models.full_non_ta_experiments import run_xgboost_full_non_ta

xgboost_metrics = run_xgboost_full_non_ta()
xgboost_metrics

## LightGBM

Data: `data-folds-full-non-ta`

Scaler: none

In [ ]:
from models.full_non_ta_experiments import run_lightgbm_full_non_ta

lightgbm_metrics = run_lightgbm_full_non_ta()
lightgbm_metrics

## AutoGluon

Data: `data-folds-full-non-ta`

Settings: `presets='medium_quality'`, `time_limit=120`, `hyperparameters='default'`, no tuning

In [ ]:
from models.full_non_ta_experiments import run_autogluon_full_non_ta

autogluon_metrics = run_autogluon_full_non_ta()
autogluon_metrics

## LSTM Sliding Windows

Data: `data-folds-full-non-ta-nn`

Scaler: `MinMaxScaler` fitted on each train fold only

Windows: `[5, 10, 20, 40, 60]`

In [ ]:
from models.full_non_ta_experiments import run_lstm_full_non_ta_all_windows

lstm_metrics_by_window = run_lstm_full_non_ta_all_windows()
lstm_metrics_by_window

## Chronos Tiny Reference

Data: `data-folds`

Feature: `Close_D` only. Chronos does not use Full Non-TA multivariate features.

In [ ]:
from models.full_non_ta_experiments import run_chronos_full_non_ta_reference

chronos_reference_metrics = run_chronos_full_non_ta_reference()
chronos_reference_metrics

## Load Saved Metrics

Loads results from `outputs/full_non_ta_feature_pool` after model cells have been run.

In [ ]:
import pandas as pd

output_root = project_root / "outputs" / "full_non_ta_feature_pool"
saved_metrics = {}
for metrics_path in sorted(output_root.glob("*/metrics_by_fold.csv")):
    saved_metrics[metrics_path.parent.name] = pd.read_csv(metrics_path)

saved_metrics